[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SheffieldML/notebook/blob/main/GPc/sampling_hmc.ipynb)

# Hybrid Monte Carlo for GP hyperparameters with `gpc`

GPc twin of [`GPy/sampling_hmc.ipynb`](../GPy/sampling_hmc.ipynb). Modelling and the leapfrog chain run in C++ ([CIP-003B](https://github.com/SheffieldML/GPc/blob/main/cip/cip003B.md)); this notebook is the NumPy / matplotlib boundary via `gpc.inference`.

Requires **`gpc>=0.1.0a11`**. Native walkthrough: [hmc_sinc](https://github.com/SheffieldML/GPc/blob/main/docs/tutorials/hmc_sinc.md).

**Differences from GPy to watch for**

* Call `gpc.inference.set_prior(m, path, "gamma", shape, rate)` (or `"gaussian", precision`) instead of `param.set_prior(GPy.priors.Gamma.from_EV(...))`. Use the `gamma_from_ev` helper below for the GPy EV parameterisation.
* `gpc.inference.hmc(...)` returns an `HmcResult` (`samples`, `names`, `accept_rate`, …). GPy returns a bare `ndarray` whose column order follows `param_array`.
* GPc includes the transform **Jacobian** in the target by default. Pass `include_jacobian=False` to match GPy’s unconstrained-space sampling without that term.
* Improper targets (a free parameter with no prior) are **refused** unless `allow_improper=True`. GPy now also expects priors on every unfixed parameter.
* No `infer_newX` / HMC over latent inputs yet — that section of the GPy twin is deferred below.
* Sample counts here are shorter than the GPy marathon chain so the notebook finishes under CI; increase `num_samples` for publication figures.


In [ ]:
import importlib
import re
import subprocess
import sys


def _gpc_ver_tuple(v: str):
    """Parse ``0.1.0a11`` / ``0.1.0`` into a comparable tuple (final > any alpha)."""
    m = re.fullmatch(r"(\d+)\.(\d+)\.(\d+)(?:a(\d+))?", str(v))
    if not m:
        return (0, 0, 0, 0)
    maj, mir, pat, alpha = m.groups()
    pre = int(alpha) if alpha is not None else 10**9
    return (int(maj), int(mir), int(pat), pre)


_MIN = "0.1.0a11"
_need = _gpc_ver_tuple(_MIN)


def _ensure_gpc():
    try:
        import gpc as _gpc
        if (
            _gpc_ver_tuple(getattr(_gpc, "__version__", "0")) >= _need
            and hasattr(_gpc, "inference")
            and hasattr(_gpc.inference, "hmc")
        ):
            return _gpc
    except ImportError:
        pass
    # Prefer TestPyPI for pre-release alphas; real PyPI for dependencies (CIP-002C).
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade",
            "--no-cache-dir",
            "--index-url",
            "https://test.pypi.org/simple/",
            "--extra-index-url",
            "https://pypi.org/simple/",
            f"gpc>={_MIN}",
        ]
    )
    import gpc as _gpc

    importlib.reload(_gpc)
    if not (hasattr(_gpc, "inference") and hasattr(_gpc.inference, "hmc")):
        raise ImportError(
            f"gpc {getattr(_gpc, '__version__', '?')} has no gpc.inference.hmc; "
            f"need >={_MIN} (CIP-003B)."
        )
    return _gpc


gpc = _ensure_gpc()
print("gpc", gpc.__version__, gpc.__file__)


In [ ]:
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import numpy as np
import gpc
from matplotlib import pyplot as plt

try:
    from gpc.plot import plot_1d
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "matplotlib"])
    from gpc.plot import plot_1d

try:
    from scipy import stats
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scipy"])
    from scipy import stats


def gamma_from_ev(E, V):
    """Match GPy ``Gamma.from_EV(E, V)`` → ``(shape, rate)`` for ``set_prior``.

    Mean ``E``, variance ``V`` ⇒ shape ``a = E²/V``, rate ``b = E/V``.
    """
    E, V = float(E), float(V)
    return E * E / V, E / V


## Example 1 — HMC for GP regression

Synthetic noisy cosine, then Gamma priors on lengthscale, signal variance, and observation noise (`likelihood.variance`).


In [ ]:
rng = np.random.default_rng(0)
x = np.linspace(0.0, 2 * np.pi, 100)[:, None]
y = -np.cos(x) + rng.normal(scale=0.3, size=x.shape) + 1.0
_ = plt.plot(x, y, ".")
plt.show()


In [ ]:
m = gpc.models.GPRegression(x, y, kernel=gpc.kern.RBF(1))
a, b = gamma_from_ev(1.0, 10.0)
for path in ("rbf.lengthscale", "rbf.variance", "likelihood.variance"):
    gpc.inference.set_prior(m, path, "gamma", a, b)
print(m)
plot_1d(m, x, y, label="before HMC (default init)")
plt.show()


### Sampling the posterior

Burn-in adapts the step size (those draws are discarded). GPy’s twin runs two separate `sample` calls; here `burn_in` covers adaptation in one call.

`include_jacobian=False` aligns the target with GPy.


In [ ]:
res = gpc.inference.hmc(
    m,
    num_samples=1000,
    burn_in=400,
    step_size=5e-2,
    num_leapfrog=20,
    seed=1,
    include_jacobian=False,
)
print("accept_rate", res.accept_rate, "adapted step_size", res.step_size)
print("columns", res.names)
s = res.samples


Trace of the chain (rows = draws, columns = parameters in `res.names`):


In [ ]:
plt.plot(s)
plt.legend(res.names)
plt.xlabel("iteration")
plt.show()


Posterior marginals (Gaussian KDE), dropping an extra head of the chain as a visual burn-in:


In [ ]:
samples = s[200:]
xmin, xmax = samples.min(), samples.max()
xs = np.linspace(xmin, xmax, 200)
for j, name in enumerate(res.names):
    kde = stats.gaussian_kde(samples[:, j])
    plt.plot(xs, kde(xs), label=name)
_ = plt.legend()
plt.show()


Pairwise scatter of posterior draws:


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
pairs = [(0, 1), (1, 2), (0, 2)]
for ax, (i, j) in zip(axes, pairs):
    ax.plot(samples[:, i], samples[:, j], ".", alpha=0.4, ms=3)
    ax.set_xlabel(res.names[i])
    ax.set_ylabel(res.names[j])
fig.tight_layout()
plt.show()


Set the model to the posterior mean and re-plot the fit (`hmc` restores parameters after sampling, so we apply the mean explicitly):


In [ ]:
means = res.mean()
for path, val in means.items():
    m.set_by_path(path, val)
print(m)
print("posterior means:", means)
plot_1d(m, x, y, label="posterior mean hyperparameters")
plt.show()


### Deferred: HMC for inputs given new outputs

The GPy twin uses `m.infer_newX(y_new)` and samples the latent inputs under a Gaussian prior on \(X\). GPc does not yet expose `infer_newX`, and `gpc.inference` Gaussian priors are zero-mean precision form — so that beat stays CLI / future-façade for now. Hyperparameter HMC on `GPRegression` / `SparseGPRegression` (and GPLVM / BGPLVM / MRD under CIP-003B) is the in-subset story.


## Example 2 — Marathon data (`pods`)

Same dataset as [`olympic_marathon.ipynb`](./olympic_marathon.ipynb) / GPy’s `olympic_marathon_men` example. Priors match the GPy twin’s `Gamma.from_EV` moments (including noise).


In [ ]:
try:
    import pods
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pods"])
    import pods

data = pods.datasets.olympic_marathon_men()
X = np.asarray(data["X"], dtype=float)
Y = np.asarray(data["Y"], dtype=float)
print("N =", X.shape[0], "year range", float(X.min()), "→", float(X.max()))

m2 = gpc.models.GPRegression(X, Y, kernel=gpc.kern.RBF(1))
# GPy: Gamma.from_EV(25, 150), (120, 2000), (1, 10) on variance / lengthscale / noise
for path, EV in (
    ("rbf.variance", (25.0, 150.0)),
    ("rbf.lengthscale", (120.0, 2000.0)),
    ("likelihood.variance", (1.0, 10.0)),
):
    a, b = gamma_from_ev(*EV)
    gpc.inference.set_prior(m2, path, "gamma", a, b)
print(m2)


### Run HMC

GPy draws 20 000 samples with `hmc_iters=20`. Here we use a shorter chain for CI; raise `num_samples` for denser seaborn plots.


In [ ]:
res2 = gpc.inference.hmc(
    m2,
    num_samples=4000,
    burn_in=800,
    step_size=2e-1,
    num_leapfrog=20,
    seed=2,
    include_jacobian=False,
)
t = res2.samples
print("accept_rate", res2.accept_rate)
print("mean", res2.mean())
print("std", {n: float(t[:, j].std()) for j, n in enumerate(res2.names)})
plt.plot(t)
plt.legend(res2.names)
plt.xlabel("iteration")
plt.show()


In [ ]:
# Histogram of kernel hyperparameters
kern_cols = [j for j, n in enumerate(res2.names) if n.startswith("rbf.")]
_ = plt.hist(t[:, kern_cols], bins=40, label=[res2.names[j] for j in kern_cols])
_ = plt.legend()
plt.show()


## Seaborn views

Optional denser marginal / joint plots (same spirit as the GPy twin).


In [ ]:
try:
    import seaborn as sns
    import pandas as pd
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "seaborn", "pandas"])
    import seaborn as sns
    import pandas as pd

plt.rcParams["text.usetex"] = False
df = pd.DataFrame(t, columns=res2.names)
var_col = next(c for c in df.columns if c.endswith("rbf.variance") or c == "rbf.variance")
ls_col = next(c for c in df.columns if c.endswith("rbf.lengthscale") or c == "rbf.lengthscale")
ax = sns.kdeplot(df[var_col], color="b", fill=True, label=var_col)
ax = sns.kdeplot(df[ls_col], color="r", fill=True, label=ls_col)
_ = ax.legend()
plt.show()


In [ ]:
sns.set(style="white", color_codes=True)
_ = sns.jointplot(data=df, x=var_col, y=ls_col, kind="hex")
plt.show()


## Takeaways

* Priors are required on every sampled path; `set_prior` + `hmc` is the façade surface (CIP-003B).
* Prefer `include_jacobian=True` (default) for priors stated in natural units; use `False` when matching older GPy HMC traces.
* `HmcResult.to_csv` matches the `gp sample` CLI layout if you want the same file as the native tutorial.
